- Topics Studied:
    - How to Improve Training Pipeline:
        - NN Module 
        - torch.optim module
    - What are the improvements:
        - not give parameters manually
        - used built in loss function
        - used built in activation function
        - not update parameter manually will use torch.optim
    - Summary:
        - Built an nueral network using nn module
        - Use built in activation function
        - Use built in loss function
        - Use built in optimizer

- The nn module:
    - nn module is an liabrary of pytorch that provide wide range of classes and functions designed to help developer to build neural network efficiently and effectively.
    - It abstracts the complexity of creating and training neural network by offering pre-built layers, loss functions, activation functions and other utilities.

- Key Components of torch.nn:
1) Modules(Layers):
    - nn.Module : The base class for all nueral network modules. Your custom models and layers should subclass this class.
    - Common Layers : Include layers like nn.Linear (fully connected layer), nn.Conv2D (convolutional layer), nn.LSTM (reccurent layer) and many others.

2) Activation Functions:
    - Functions like nn.ReLU, nn.Sigmoid and nn.Tanh itroduce non-linearities to the model, allowing it to learn complex patterns.

3) Loss Functions:
    - Provide loss functions such as nn.CrossEntropyLoss, nn,MSELoss and nn.NLLLoss to quanify the differnce between the model's predictions and the actual targets/

4) Container Modules:
    - nn.Sequential : A sequential container to stack layers in orders.

5) Regularization and Dropout:
    - Layer like nn.Dropout and nn.BatchNorm1d help prevent overfitting and improves the model's ability to generalize new data. 

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
df = pd.read_csv("breast_cancer_data.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.drop(columns=["id","Unnamed: 32"],inplace=True)

In [ ]:
df.head()

train test split

In [ ]:
x = df.drop(columns="diagnosis")
y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

label encoding

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

array to tensor

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

defining model

In [ ]:
class MySimpleNN():
    def __init__(self,X):
        self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)
        
    def forward(self,X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self, y_pred,y):
        # clamp prediction to avoide log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)

        # calculate loss
        loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor)* torch.log(1-y_pred)).mean()
        return loss


important parameters

In [ ]:
learning_rate = 0.1
epochs = 35

training pipeline

In [ ]:
# create model
model = MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):
    # forward pass
    y_pred = model.forward(X_train_tensor)
    
    # loss calculate
    loss = model.loss_function(y_pred, y_train_tensor)

    # backward pass
    loss.backward()

    # parameters update
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # print loss in each epoch
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")